# 02 - Monitoramento de Fluxo com IoT
> Backend, dashboard web e integracao com sensores

**Modulo:** `13_enfitec_projetos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Arquitetura do Sistema IoT

Vamos pedir ao Claude para projetar a arquitetura completa do sistema
de monitoramento de fluxo: desde o sensor ate o dashboard.

In [ ]:
prompt_arq = """Projete a arquitetura completa de um sistema IoT para
monitoramento de fluxo de agua em uma industria. O sistema deve incluir:

1. CAMADA SENSOR: Sensor de fluxo (tipo turbina ou eletromagnetico)
   conectado a um ESP32 via GPIO
2. CAMADA COMUNICACAO: ESP32 envia dados via MQTT para um broker
   (Mosquitto)
3. CAMADA BACKEND: Servidor Python (Flask ou FastAPI) que:
   - Recebe dados do MQTT
   - Armazena em banco de dados (SQLite para prototipo)
   - Fornece API REST para o dashboard
   - Processa alertas de fluxo fora da faixa
4. CAMADA FRONTEND: Dashboard web mostrando:
   - Fluxo em tempo real (grafico de linha)
   - Volume acumulado
   - Alertas ativos
   - Historico

Desenhe a arquitetura em formato texto/ASCII e descreva cada componente,
protocolo de comunicacao, formato de dados (JSON) e frequencia de leitura."""

resp_arq = ask(prompt_arq, model=SONNET, max_tokens=2048)
print(resp_arq)

In [ ]:
# Diagrama de arquitetura em texto
arquitetura = """
=======================================================================
            ARQUITETURA DO SISTEMA DE MONITORAMENTO DE FLUXO
=======================================================================

[SENSOR DE FLUXO]  --(pulsos)--> [ESP32]
  YF-S201 / DN25                   |
  1-30 L/min                       | WiFi / MQTT
  Pulsos: 7.5 * Q                  | Topico: planta/fluxo/sensor01
                                   | QoS: 1
                                   v
                          [BROKER MOSQUITTO]
                            porta 1883
                                   |
                                   v
                      [BACKEND FastAPI / Python]
                       +------------------------+
                       | Subscriber MQTT        |
                       | API REST (:8000)       |
                       | Processamento alertas  |
                       | SQLite (dados.db)      |
                       +------------------------+
                                   |
                            API REST / JSON
                                   v
                      [DASHBOARD Plotly Dash]
                       +------------------------+
                       | Grafico tempo real      |
                       | Volume acumulado        |
                       | Painel de alertas       |
                       | Historico + exportacao  |
                       +------------------------+

FORMATO JSON (MQTT):
{
    "sensor_id": "fluxo_01",
    "timestamp": "2025-01-15T10:30:00Z",
    "flow_rate": 12.5,       // L/min
    "total_volume": 1523.7,  // Litros
    "temperature": 23.4      // Celsius (opcional)
}

FREQUENCIA DE LEITURA: 1 Hz (1 leitura/segundo)
ENVIO MQTT: a cada 5 segundos (media de 5 leituras)
=======================================================================
"""
print(arquitetura)

## Firmware ESP32 para Sensor de Fluxo

Vamos pedir ao Claude para gerar firmware completo para o ESP32
que le um sensor de fluxo e envia dados via MQTT.

In [ ]:
prompt_firmware = """Gere firmware completo em C/C++ (Arduino framework) para ESP32
que faz o seguinte:

1. Le um sensor de fluxo tipo YF-S201 no pino GPIO 27
   - Usa interrupcao para contar pulsos
   - Converte pulsos em vazao (L/min): Q = pulsos / (7.5 * intervalo_s)
   - Calcula volume total acumulado

2. Conecta ao WiFi (SSID e senha configuraveis)

3. Publica dados via MQTT a cada 5 segundos:
   - Broker: configuravel
   - Topico: "planta/fluxo/sensor01"
   - Formato JSON com: sensor_id, timestamp, flow_rate, total_volume

4. Funcionalidades extras:
   - LED indicador de status (conectado/desconectado)
   - Reconexao automatica WiFi e MQTT
   - Watchdog timer
   - Deep sleep se sem WiFi por mais de 5 minutos

Use as bibliotecas: WiFi.h, PubSubClient.h, ArduinoJson.h.
Retorne o codigo completo com comentarios em portugues."""

resp_firmware = ask(prompt_firmware, model=SONNET, max_tokens=4096)
print(resp_firmware)

In [ ]:
# O firmware gerado pelo Claude pode ser salvo em arquivo .ino
firmware_exemplo = '''// =============================================
// Firmware ESP32 - Sensor de Fluxo com MQTT
// =============================================
// Gerado com auxilio do Claude (Anthropic)

#include <WiFi.h>
#include <PubSubClient.h>
#include <ArduinoJson.h>

// --- Configuracoes ---
#define FLOW_PIN      27
#define LED_PIN       2
#define MQTT_INTERVAL 5000  // ms
#define PULSE_FACTOR  7.5   // pulsos por litro/min

const char* WIFI_SSID     = "SUA_REDE";
const char* WIFI_PASS     = "SUA_SENHA";
const char* MQTT_BROKER   = "192.168.1.100";
const int   MQTT_PORT     = 1883;
const char* MQTT_TOPIC    = "planta/fluxo/sensor01";
const char* SENSOR_ID     = "fluxo_01";

// --- Variaveis Globais ---
volatile unsigned long pulseCount = 0;
float flowRate = 0.0;
float totalVolume = 0.0;
unsigned long lastPublish = 0;
unsigned long lastPulseCheck = 0;

WiFiClient espClient;
PubSubClient mqtt(espClient);

// --- ISR para contagem de pulsos ---
void IRAM_ATTR pulseCounter() {
    pulseCount++;
}

void setup() {
    Serial.begin(115200);
    pinMode(FLOW_PIN, INPUT_PULLUP);
    pinMode(LED_PIN, OUTPUT);
    attachInterrupt(digitalPinToInterrupt(FLOW_PIN),
                    pulseCounter, FALLING);

    // Conectar WiFi
    WiFi.begin(WIFI_SSID, WIFI_PASS);
    while (WiFi.status() != WL_CONNECTED) {
        delay(500);
        Serial.print(".");
    }
    Serial.println("WiFi conectado!");

    // Configurar MQTT
    mqtt.setServer(MQTT_BROKER, MQTT_PORT);
}

void loop() {
    if (!mqtt.connected()) reconnectMQTT();
    mqtt.loop();

    unsigned long now = millis();

    if (now - lastPublish >= MQTT_INTERVAL) {
        // Calcular vazao
        noInterrupts();
        unsigned long pulses = pulseCount;
        pulseCount = 0;
        interrupts();

        float elapsed = (now - lastPulseCheck) / 1000.0;
        flowRate = pulses / (PULSE_FACTOR * elapsed);
        totalVolume += flowRate * elapsed / 60.0;

        // Publicar JSON
        StaticJsonDocument<256> doc;
        doc["sensor_id"] = SENSOR_ID;
        doc["flow_rate"] = flowRate;
        doc["total_volume"] = totalVolume;
        doc["timestamp"] = now;

        char buffer[256];
        serializeJson(doc, buffer);
        mqtt.publish(MQTT_TOPIC, buffer);

        Serial.printf("Vazao: %.2f L/min | Total: %.2f L\\n",
                      flowRate, totalVolume);

        lastPublish = now;
        lastPulseCheck = now;
        digitalWrite(LED_PIN, !digitalRead(LED_PIN));
    }
}

void reconnectMQTT() {
    while (!mqtt.connected()) {
        Serial.print("Conectando MQTT...");
        if (mqtt.connect(SENSOR_ID)) {
            Serial.println("conectado!");
        } else {
            Serial.printf("falhou (rc=%d). Tentando em 5s...\\n",
                          mqtt.state());
            delay(5000);
        }
    }
}
'''

print('Firmware de exemplo carregado.')
print(f'Tamanho: {len(firmware_exemplo)} caracteres')
print('\nPara salvar: copie a variavel firmware_exemplo para um arquivo .ino')

## Backend com Flask/FastAPI

Vamos pedir ao Claude para gerar uma API completa que recebe dados
dos sensores e armazena em SQLite.

In [ ]:
prompt_backend = """Gere codigo Python completo para um backend FastAPI que:

1. Recebe dados de sensor de fluxo via:
   - POST /api/readings (JSON: sensor_id, flow_rate, total_volume, timestamp)
   - Subscriber MQTT (topico: planta/fluxo/+)

2. Armazena em SQLite com tabelas:
   - sensors (id, name, location, min_flow, max_flow)
   - readings (id, sensor_id, flow_rate, total_volume, timestamp)
   - alerts (id, sensor_id, alert_type, message, timestamp, acknowledged)

3. Fornece endpoints REST:
   - GET /api/sensors - listar sensores
   - GET /api/readings/{sensor_id}?start=...&end=... - leituras por periodo
   - GET /api/readings/{sensor_id}/latest - ultima leitura
   - GET /api/alerts - alertas ativos
   - POST /api/alerts/{id}/acknowledge - reconhecer alerta
   - GET /api/stats/{sensor_id} - estatisticas (media, max, min, volume total)

4. Processa alertas automaticamente:
   - Fluxo abaixo do minimo (possivel vazamento ou obstrucao)
   - Fluxo acima do maximo (sobrepressao)
   - Sensor offline (sem dados por mais de 30 segundos)

Use: fastapi, uvicorn, sqlite3, pydantic, paho-mqtt.
Retorne codigo completo e funcional."""

resp_backend = ask(prompt_backend, model=SONNET, max_tokens=4096)
print(resp_backend)

In [ ]:
# Exemplo minimo do backend para demonstracao
backend_code = '''
# backend_fluxo.py - API de Monitoramento de Fluxo
# Gerado com auxilio do Claude (Anthropic)

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from datetime import datetime
import sqlite3
import json

app = FastAPI(title="API Monitoramento de Fluxo", version="1.0.0")

# --- Modelos Pydantic ---
class Reading(BaseModel):
    sensor_id: str
    flow_rate: float
    total_volume: float
    timestamp: str = None

class Sensor(BaseModel):
    name: str
    location: str
    min_flow: float = 0.5
    max_flow: float = 25.0

# --- Banco de Dados ---
def get_db():
    conn = sqlite3.connect("fluxo_data.db")
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db()
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS sensors (
            id TEXT PRIMARY KEY, name TEXT,
            location TEXT, min_flow REAL, max_flow REAL
        );
        CREATE TABLE IF NOT EXISTS readings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sensor_id TEXT, flow_rate REAL,
            total_volume REAL, timestamp TEXT
        );
        CREATE TABLE IF NOT EXISTS alerts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sensor_id TEXT, alert_type TEXT,
            message TEXT, timestamp TEXT,
            acknowledged INTEGER DEFAULT 0
        );
    """)
    conn.close()

@app.on_event("startup")
async def startup():
    init_db()

# --- Endpoints ---
@app.post("/api/readings")
async def add_reading(reading: Reading):
    if not reading.timestamp:
        reading.timestamp = datetime.utcnow().isoformat()
    conn = get_db()
    conn.execute(
        "INSERT INTO readings (sensor_id, flow_rate, total_volume, timestamp) "
        "VALUES (?, ?, ?, ?)",
        (reading.sensor_id, reading.flow_rate,
         reading.total_volume, reading.timestamp)
    )
    check_alerts(conn, reading)
    conn.commit()
    conn.close()
    return {"status": "ok"}

@app.get("/api/readings/{sensor_id}/latest")
async def latest_reading(sensor_id: str):
    conn = get_db()
    row = conn.execute(
        "SELECT * FROM readings WHERE sensor_id = ? "
        "ORDER BY timestamp DESC LIMIT 1", (sensor_id,)
    ).fetchone()
    conn.close()
    if not row:
        raise HTTPException(404, "Sensor nao encontrado")
    return dict(row)

@app.get("/api/alerts")
async def get_alerts():
    conn = get_db()
    rows = conn.execute(
        "SELECT * FROM alerts WHERE acknowledged = 0 "
        "ORDER BY timestamp DESC"
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def check_alerts(conn, reading):
    sensor = conn.execute(
        "SELECT * FROM sensors WHERE id = ?", (reading.sensor_id,)
    ).fetchone()
    if not sensor:
        return
    if reading.flow_rate < sensor["min_flow"]:
        conn.execute(
            "INSERT INTO alerts (sensor_id, alert_type, message, timestamp) "
            "VALUES (?, ?, ?, ?)",
            (reading.sensor_id, "LOW_FLOW",
             f"Fluxo baixo: {reading.flow_rate:.2f} L/min",
             reading.timestamp)
        )
    elif reading.flow_rate > sensor["max_flow"]:
        conn.execute(
            "INSERT INTO alerts (sensor_id, alert_type, message, timestamp) "
            "VALUES (?, ?, ?, ?)",
            (reading.sensor_id, "HIGH_FLOW",
             f"Fluxo alto: {reading.flow_rate:.2f} L/min",
             reading.timestamp)
        )

# Executar: uvicorn backend_fluxo:app --reload --port 8000
'''

print('Backend de exemplo carregado.')
print(f'Tamanho: {len(backend_code)} caracteres')
print('\nEndpoints disponiveis:')
print('  POST /api/readings')
print('  GET  /api/readings/{sensor_id}/latest')
print('  GET  /api/alerts')

## Dashboard com Plotly/Dash

Vamos pedir ao Claude para gerar um dashboard em tempo real
mostrando vazao, alertas e estatisticas.

In [ ]:
prompt_dash = """Gere codigo Python completo para um dashboard usando Plotly Dash
para monitoramento de fluxo de agua em tempo real. O dashboard deve ter:

1. HEADER: Titulo do sistema + indicador de status da conexao

2. PAINEL PRINCIPAL (linha 1):
   - Card com vazao atual (numero grande, cor muda conforme status)
   - Card com volume acumulado hoje
   - Card com numero de alertas ativos
   - Card com uptime do sensor

3. GRAFICOS (linha 2):
   - Grafico de linha: vazao vs tempo (ultimas 2 horas)
     com faixas de operacao normal (verde) e alerta (vermelho)
   - Grafico de barras: volume por hora (ultimas 24h)

4. TABELA DE ALERTAS (linha 3):
   - Ultimos 10 alertas com: timestamp, tipo, mensagem, status
   - Botao para reconhecer cada alerta

5. ATUALIZACAO:
   - Intervalo de atualizacao: 5 segundos (dcc.Interval)
   - Busca dados via API REST do backend

Use: dash, plotly, dash-bootstrap-components.
Retorne codigo completo e funcional com layout responsivo."""

resp_dash = ask(prompt_dash, model=SONNET, max_tokens=4096)
print(resp_dash)

In [ ]:
dashboard_code = '''
# dashboard_fluxo.py - Dashboard de Monitoramento de Fluxo
# Gerado com auxilio do Claude (Anthropic)

import dash
from dash import html, dcc, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import requests
from datetime import datetime, timedelta

API_BASE = "http://localhost:8000/api"

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

# --- Cards de Status ---
def make_card(title, value, color="primary"):
    return dbc.Card([
        dbc.CardBody([
            html.H6(title, className="card-title text-muted"),
            html.H2(value, className=f"text-{color}"),
        ])
    ], className="mb-3")

# --- Layout ---
app.layout = dbc.Container([
    dcc.Interval(id="interval", interval=5000),

    # Header
    dbc.Row(dbc.Col(html.H2(
        "Monitoramento de Fluxo - Planta Industrial",
        className="text-center my-3"
    ))),

    # Cards
    dbc.Row([
        dbc.Col(html.Div(id="card-vazao"), width=3),
        dbc.Col(html.Div(id="card-volume"), width=3),
        dbc.Col(html.Div(id="card-alertas"), width=3),
        dbc.Col(html.Div(id="card-uptime"), width=3),
    ]),

    # Graficos
    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico-vazao"), width=8),
        dbc.Col(dcc.Graph(id="grafico-volume"), width=4),
    ]),

    # Tabela de Alertas
    dbc.Row(dbc.Col([
        html.H4("Alertas Recentes"),
        html.Div(id="tabela-alertas")
    ])),
], fluid=True)

@app.callback(
    [Output("card-vazao", "children"),
     Output("grafico-vazao", "figure")],
    [Input("interval", "n_intervals")]
)
def update_dashboard(n):
    try:
        resp = requests.get(f"{API_BASE}/readings/fluxo_01/latest")
        data = resp.json()
        vazao = data.get("flow_rate", 0)
        cor = "success" if 2 < vazao < 20 else "danger"
        card = make_card("Vazao Atual (L/min)", f"{vazao:.1f}", cor)
    except Exception:
        card = make_card("Vazao Atual", "Offline", "secondary")
        vazao = 0

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="lines", name="Vazao"
    ))
    fig.update_layout(
        template="plotly_dark",
        title="Vazao vs Tempo",
        xaxis_title="Tempo",
        yaxis_title="Vazao (L/min)"
    )

    return card, fig

# Executar: python dashboard_fluxo.py
if __name__ == "__main__":
    app.run(debug=True, port=8050)
'''

print('Dashboard de exemplo carregado.')
print(f'Tamanho: {len(dashboard_code)} caracteres')
print('\nPara executar:')
print('  pip install dash plotly dash-bootstrap-components requests')
print('  python dashboard_fluxo.py')
print('  Abrir: http://localhost:8050')

## Integracao Completa

Vamos pedir ao Claude para integrar todos os componentes
em um sistema coeso com tratamento de erros.

In [ ]:
prompt_integracao = """Temos os seguintes componentes de um sistema IoT de
monitoramento de fluxo:

1. ESP32 com sensor YF-S201 enviando dados via MQTT
2. Backend FastAPI recebendo dados e armazenando em SQLite
3. Dashboard Plotly Dash consumindo API REST

Gere o codigo de integracao que:

a) docker-compose.yml para subir todos os servicos:
   - Mosquitto (broker MQTT)
   - Backend FastAPI
   - Dashboard Dash
   - (opcional) Simulador de sensor para testes

b) Script simulador_sensor.py que simula o ESP32:
   - Gera dados realistas de fluxo (variacao senoidal + ruido)
   - Injeta anomalias aleatorias
   - Publica via MQTT no mesmo formato do ESP32

c) Script de health-check que verifica:
   - Broker MQTT online
   - Backend respondendo
   - Dashboard acessivel
   - Sensores enviando dados

d) Tratamento robusto de erros em todos os componentes:
   - Reconexao automatica
   - Logging estruturado
   - Graceful shutdown

Retorne o codigo completo de cada componente."""

resp_integracao = ask(prompt_integracao, model=SONNET, max_tokens=4096)
print(resp_integracao)

In [ ]:
# Estrutura final do projeto
estrutura = """
monitoramento-fluxo/
|
|-- docker-compose.yml
|-- .env
|
|-- firmware/
|   |-- sensor_fluxo.ino
|   |-- config.h
|   |-- platformio.ini
|
|-- backend/
|   |-- main.py              # FastAPI app
|   |-- models.py            # Modelos Pydantic
|   |-- database.py          # Conexao SQLite
|   |-- mqtt_subscriber.py   # Cliente MQTT
|   |-- alert_engine.py      # Motor de alertas
|   |-- requirements.txt
|   |-- Dockerfile
|
|-- dashboard/
|   |-- app.py               # Dash app
|   |-- layouts.py           # Componentes de layout
|   |-- callbacks.py         # Callbacks Dash
|   |-- requirements.txt
|   |-- Dockerfile
|
|-- tools/
|   |-- simulador_sensor.py  # Simulador para testes
|   |-- health_check.py      # Verificacao de saude
|   |-- seed_db.py           # Popular banco com dados iniciais
|
|-- mosquitto/
|   |-- mosquitto.conf
|
|-- README.md
"""

print('Estrutura do projeto:')
print(estrutura)
print('Todos os componentes foram gerados com auxilio do Claude.')
print('O proximo passo e implementar e testar cada modulo individualmente.')

## Exercicios

1. **Multiplos Sensores:** Modifique o backend para suportar ate 10 sensores
   simultaneos. Cada sensor deve ter seus proprios limiares de alerta.

2. **Persistencia com PostgreSQL:** Substitua o SQLite por PostgreSQL
   usando SQLAlchemy como ORM. Adicione migrations com Alembic.

3. **Autenticacao:** Adicione autenticacao JWT ao backend. O dashboard
   deve exigir login para acesso.

4. **Notificacoes:** Implemente envio de alertas por e-mail (SMTP)
   e Telegram (bot API) quando um alerta critico for disparado.

5. **OTA Update:** Peca ao Claude para adicionar capacidade de
   atualizacao Over-The-Air (OTA) ao firmware do ESP32.

## Proximos Passos

- Adicionar protocolo LoRaWAN para sensores em locais sem WiFi
- Implementar HTTPS e certificados TLS para MQTT
- Criar app mobile com React Native para monitoramento remoto
- Adicionar machine learning para previsao de consumo de agua
- Integrar com sistemas SCADA existentes via OPC-UA
- Implementar redundancia no backend (load balancer + replicas)

---

*Notebook criado com auxilio do Claude (Anthropic) para o modulo 13 - Enfitec Projetos.*